# Peramalan Tekanan Sistem Pangan ASEAN (1961–2024 → 2035)
### RASIO 10.0 — Padjadjaran Statistics Olympiad · panel pembuka infografis

**Unit analisis:** 64 titik tahunan, 1961–2024 · **Keluaran:** proyeksi 11 tahun + SVG vektor
**Sumber:** Our World in Data — *CO2 and Greenhouse Gas Emissions* (Global Carbon Budget; Jones dkk.)

| Variabel | Deskripsi | Peran |
|---|---|---|
| `ch4` | Metana ASEAN — sawah tergenang & ternak ruminansia | **Target utama** |
| `n2o` | N₂O ASEAN — intensitas pemupukan nitrogen | Target sekunder |
| `pangsa_ch4` | Pangsa ASEAN dalam metana dunia (%) | Target sekunder |
| `kum_luc` | Kumulatif CO₂ alih guna lahan | Target sekunder |
| `populasi_jt` | Populasi ASEAN (juta jiwa) | Kandidat regressor eksogen |
| `co2` | CO₂ energi ASEAN | Kandidat regressor eksogen |
| `luc` | CO₂ alih guna lahan tahunan | **Bukan** regressor — lihat Bagian 3 |

**Perbedaan konteks vs notebook kesehatan mental (bulanan).** Data ini **tahunan**, sehingga
tidak ada komponen musiman: STL 12-bulanan, SARIMA musiman, dan seasonal-naive tidak berlaku
dan diganti dekomposisi tren LOESS, ARIMA non-musiman, serta baseline naif/drift. Sebagai
gantinya, kedalaman dipindahkan ke tiga hal yang tidak ada di notebook bulanan: **uji
Diebold-Mariano** untuk menguji apakah selisih akurasi antarmodel signifikan secara statistik,
**kalibrasi selang prediksi** untuk memeriksa apakah pita 80%/95% benar-benar memuat 80%/95%
kejadian, dan **rekonsiliasi hirarkis** antara ramalan agregat ASEAN dan penjumlahan ramalan
sepuluh negara anggotanya.

**Catatan implementasi.** Seluruh prosedur ditulis ulang dengan numpy/scipy sehingga notebook
berjalan tanpa statsmodels maupun prophet. Bila paket itu tersedia, Bagian 18 memuat sel
pembanding untuk memverifikasi angkanya.

**Berkas pendamping:** `agregat_asean_tahunan.csv`, `metana_per_negara_asean.csv`

## 0. Setup & Dependencies

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor

RS = 42; np.random.seed(RS)
sns.set_theme(style='whitegrid')
PALETTE = {'primary':'#2F7D6E','secondary':'#3A8C7E','accent':'#D9B26F',
           'danger':'#C0392B','purple':'#6C5B9E','muted':'#8896A0',
           'low':'#BFE0D8','dark':'#243B3A'}
plt.rcParams.update({'figure.dpi':110,'savefig.bbox':'tight','font.size':9,
                     'axes.titlesize':10.5})
DIR = '.'; SVG_DIR = 'gambar_forecast'
os.makedirs(SVG_DIR, exist_ok=True)
print('Setup selesai.')

## 1. Memuat Data

In [ ]:
a = pd.read_csv(f'{DIR}/agregat_asean_tahunan.csv', index_col=0)
neg = pd.read_csv(f'{DIR}/metana_per_negara_asean.csv', index_col=0)

KOLOM = ['ch4','n2o','pangsa_ch4','kum_luc','populasi_jt','co2','luc']
LABEL = {'ch4':'Metana ASEAN (Mt CO2e)','n2o':'N2O ASEAN (Mt CO2e)',
         'pangsa_ch4':'Pangsa Metana Global (%)','kum_luc':'Kumulatif CO2 Alih Guna Lahan (Mt)',
         'populasi_jt':'Populasi ASEAN (juta)','co2':'CO2 Energi ASEAN (Mt)',
         'luc':'CO2 Alih Guna Lahan Tahunan (Mt)'}
Y_COL = 'ch4'

print('Periode :', a.index.min(), '-', a.index.max())
print('N       :', len(a), '| data hilang:', int(a[KOLOM].isna().sum().sum()))
print('Negara  :', ', '.join(neg.columns))
display(a[KOLOM].describe().T.round(2))
a.tail(3)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
for ax, c, w in zip(axes, ['ch4','n2o','pangsa_ch4','luc'],
                    [PALETTE['primary'], PALETTE['accent'], PALETTE['purple'], PALETTE['danger']]):
    ax.plot(a.index, a[c], color=w, lw=1.9)
    ax.set_title(LABEL[c], fontsize=10.5, loc='left')
axes[3].text(1998, a['luc'].max()*.93,
             'perhatikan gejolak tahun kebakaran gambut\n(1997-98, 2015) -- alasan seri ini\n'
             'TIDAK dijadikan target ramalan',
             fontsize=8, color=PALETTE['muted'])
fig.suptitle('Empat Indikator Tekanan Sistem Pangan ASEAN, 1961-2024', fontweight='bold', y=.995)
plt.tight_layout()
fig.savefig(f'{SVG_DIR}/00_empat_seri.svg', format='svg', bbox_inches='tight'); plt.show()

## 2. Dekomposisi Tren dan Stasioneritas

Data tahunan tidak memiliki komponen musiman, sehingga STL periode-12 tidak berlaku. Sebagai
gantinya tren diekstraksi dengan **LOESS**, lalu kekuatan tren dihitung dengan definisi yang
sama seperti pada dekomposisi musiman. Orde differencing ditetapkan lewat dua uji yang
hipotesis nolnya berlawanan — ADF (H₀: ada akar unit) dan KPSS (H₀: stasioner) — agar
kesimpulannya tidak bergantung pada satu uji saja.

In [ ]:
def _ols(X, y):
    b, *_ = np.linalg.lstsq(X, y, rcond=None); return b

def loess_tren(y, frac=0.30):
    y = np.asarray(y, float); n = len(y); x = np.arange(n); out = np.empty(n)
    r = max(int(frac*n), 3)
    for i in range(n):
        d = np.abs(x-i); h = np.sort(d)[min(r, n-1)]
        w = np.clip(1-(d/max(h,1e-9))**3, 0, None)**3
        X = np.column_stack([np.ones(n), x-i]); WX = X*w[:,None]
        out[i] = _ols(WX.T@X, WX.T@y)[0]
    return out

ly = np.log(a[Y_COL].values)
tren = loess_tren(ly); sisa = ly - tren
kekuatan_tren = max(0, 1 - np.var(sisa)/np.var(tren+sisa))

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(a.index, ly, color=PALETTE['dark'], lw=1.8); axes[0].set_title('log(Metana ASEAN)', loc='left')
axes[1].plot(a.index, tren, color=PALETTE['primary'], lw=2); axes[1].set_title('Komponen tren (LOESS)', loc='left')
axes[2].bar(a.index, sisa, color=PALETTE['muted']); axes[2].axhline(0, color='grey', lw=.8)
axes[2].set_title('Sisa', loc='left')
fig.suptitle('Dekomposisi Tren-Sisa (tanpa komponen musiman: data tahunan)', fontweight='bold', y=1.0)
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/01_dekomposisi_tren.svg', format='svg', bbox_inches='tight'); plt.show()
print(f'Kekuatan tren = {kekuatan_tren:.3f}  (>0.6 = tren kuat; data tahunan -> musiman tidak diukur)')

In [ ]:
_ADF_CV = {'1%':-3.51, '5%':-2.89, '10%':-2.58}
_KPSS_CV = {'1%':0.739, '5%':0.463, '10%':0.347}

def adf(y, maks_lag=4):
    y = np.asarray(y, float); best = None
    for L in range(maks_lag+1):
        dy = np.diff(y); n = len(dy)-L
        if n < 15: continue
        X = [np.ones(n), y[L:-1]]
        for i in range(1, L+1): X.append(dy[L-i:-i])
        X = np.column_stack(X); yy = dy[L:]
        b = _ols(X, yy); e = yy - X@b; k = X.shape[1]
        se = np.sqrt(np.diag(((e@e)/(n-k))*np.linalg.pinv(X.T@X)))
        aic = n*np.log((e@e)/n) + 2*k
        c = (aic, b[1]/se[1], L)
        if best is None or c[0] < best[0]: best = c
    return best[1], best[2]

def kpss(y, lags=None):
    y = np.asarray(y, float); n = len(y); e = y - y.mean(); S = np.cumsum(e)
    if lags is None: lags = int(4*(n/100)**0.25)
    s2 = (e@e)/n
    for L in range(1, lags+1):
        s2 += 2*(1-L/(lags+1))*np.sum(e[L:]*e[:-L])/n
    return (np.sum(S**2)/(n**2))/s2

def uji_stasioner(s, nama):
    st, lag = adf(s); kp = kpss(s)
    print(f'{nama:26s} ADF={st:7.3f} ({"stasioner" if st < _ADF_CV["5%"] else "NON-stasioner"}, lag={lag})'
          f'  | KPSS={kp:6.4f} ({"stasioner" if kp < _KPSS_CV["5%"] else "NON-stasioner"})')

print(f'Nilai kritis 5%: ADF {_ADF_CV["5%"]}, KPSS {_KPSS_CV["5%"]}\n')
uji_stasioner(ly, 'Level (log)')
uji_stasioner(np.diff(ly), 'Differencing d=1')
uji_stasioner(np.diff(np.diff(ly)), 'Differencing d=2')
D_ORDER = 1
print(f'\nOrde differencing dipakai: d={D_ORDER}')

In [ ]:
def acf(y, nlags=20):
    y = np.asarray(y,float) - np.mean(y); d = y@y
    return np.array([1.0] + [(y[k:]@y[:-k])/d for k in range(1, nlags+1)])

def pacf(y, nlags=20):
    r = acf(y, nlags); out = [1.0]
    for k in range(1, nlags+1):
        R = np.array([[r[abs(i-j)] for j in range(k)] for i in range(k)])
        out.append(_ols(R, r[1:k+1])[-1])
    return np.array(out)

d1 = np.diff(ly); L = 20; band = 1.96/np.sqrt(len(d1))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, f, t in zip(axes, [acf, pacf], ['ACF (setelah d=1)','PACF (setelah d=1)']):
    v = f(d1, L)
    ax.bar(range(L+1), v, color=PALETTE['primary'], width=.35)
    ax.axhline(0, color='grey', lw=.8)
    ax.axhline(band, color=PALETTE['danger'], ls='--', lw=.9)
    ax.axhline(-band, color=PALETTE['danger'], ls='--', lw=.9)
    ax.set_title(t); ax.set_xlabel('Lag')
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/02_acf_pacf.svg', format='svg', bbox_inches='tight'); plt.show()
print('Lag signifikan ACF :', [i for i,v in enumerate(acf(d1,L)) if i>0 and abs(v)>band])
print('Lag signifikan PACF:', [i for i,v in enumerate(pacf(d1,L)) if i>0 and abs(v)>band])

## 3. Korelasi Antarindikator dan Diagnostik Kebocoran Data

Persoalannya sama seperti pada notebook kesehatan mental: korelasi kontemporer yang tinggi
belum tentu berguna untuk meramal, karena nilainya baru diketahui pada tahun yang sama dengan
target. CCF dihitung pada seri yang sudah dibuang trennya agar tren bersama tidak mendominasi.

In [ ]:
KOR = ['ch4','n2o','luc','co2','populasi_jt']
kor = a[KOR].corr()
plt.figure(figsize=(6.4, 5))
sns.heatmap(kor, annot=True, fmt='.3f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, xticklabels=[LABEL[c][:22] for c in KOR],
            yticklabels=[LABEL[c][:22] for c in KOR])
plt.title('Korelasi Pearson Antarindikator (level)')
plt.tight_layout(); plt.show()

def ccf_dibuang_tren(x, y, maks_lag=10):
    xs, ys = pd.Series(x).diff().dropna(), pd.Series(y).diff().dropna()
    xs, ys = xs.align(ys, join='inner')
    return pd.Series({lag: xs.corr(ys.shift(-lag)) for lag in range(-maks_lag, maks_lag+1)})

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
amb = 1.96/np.sqrt(len(a)-1)
for ax, drv in zip(axes, ['populasi_jt','co2','luc']):
    c = ccf_dibuang_tren(a[drv].values, a[Y_COL].values)
    ax.bar(c.index, c.values,
           color=[PALETTE['danger'] if abs(v) > amb else PALETTE['primary'] for v in c.values])
    ax.axhline(amb, color=PALETTE['muted'], ls='--', lw=.8)
    ax.axhline(-amb, color=PALETTE['muted'], ls='--', lw=.8)
    ax.axvline(0, color='grey', lw=.8)
    ax.set_title(f'CCF: {LABEL[drv][:20]} -> metana', fontsize=9.5)
    ax.set_xlabel('lag (positif = driver mendahului)')
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/03_ccf_kebocoran.svg', format='svg', bbox_inches='tight'); plt.show()

**Keputusan regressor eksogen.**

- `luc` (CO₂ alih guna lahan tahunan) **dikeluarkan**. Ia sepenuhnya kontemporer dengan target
  dan nilainya baru diketahui pada tahun yang sama, sehingga memakainya untuk meramal tahun itu
  adalah kebocoran data. Lebih jauh, uji Bagian 15 menunjukkan seri ini sendiri tidak layak
  diramal (MAPE > 23%) karena gejolak tahun kebakaran gambut — memakainya sebagai regressor
  berarti memindahkan ketidakpastian itu ke dalam model target.
- `populasi_jt` **dipertahankan**. Ini satu-satunya regressor yang tersedia lebih dulu secara
  operasional: proyeksi populasi PBB dirilis independen dan jauh ke depan, sehingga nilai masa
  depannya diketahui tanpa perlu diramal sendiri — properti yang jarang dimiliki regressor.
- `co2` **dipertahankan sebagai kandidat**, tetapi nilai tambahnya diuji lewat CV pada Bagian 6,
  bukan diasumsikan.

## 4. Fungsi Bantu dan Metrik Evaluasi

In [ ]:
def mape(t, p):  t, p = np.asarray(t,float), np.asarray(p,float); return np.mean(np.abs((t-p)/t))*100
def rmse(t, p):  t, p = np.asarray(t,float), np.asarray(p,float); return np.sqrt(np.mean((t-p)**2))
def smape(t, p): t, p = np.asarray(t,float), np.asarray(p,float); return np.mean(2*np.abs(t-p)/(np.abs(t)+np.abs(p)))*100
def mase(t, p, train):
    t, p = np.asarray(t,float), np.asarray(p,float)
    d = np.mean(np.abs(np.diff(np.asarray(train,float))))
    return np.mean(np.abs(t-p))/d if d > 0 else np.nan

print('MASE ditambahkan karena bebas skala: penyebutnya adalah MAE naive satu-langkah pada data')
print('latih, sehingga model pada seri berbeda (metana, N2O, pangsa) dapat dibandingkan setara.')
print('Tafsir: MASE < 1 pada h=1 berarti mengungguli naive. Nilainya MENINGKAT seiring horizon')
print('secara konstruksi -- penyebutnya selalu galat satu-langkah -- jadi MASE h=10 tidak boleh')
print('dibandingkan langsung dengan ambang 1, melainkan antar-model pada horizon yang sama.')

## 5. Definisi Model Kandidat

Tiga belas kandidat diuji — hampir dua kali jumlah pada notebook bulanan. Kelompoknya:
**baseline** (naive, drift, rata-rata bergerak), **pemulusan eksponensial** (Holt, Holt teredam,
Holt dengan parameter dituning lewat SSE), **klasik** (Theta, ARIMA dengan orde dipilih AICc,
ARIMAX dengan regressor eksogen), **regresi** (tren linear, Ridge atas fitur lag), **mesin
pembelajaran** (Gradient Boosting atas fitur lag), dan **ensemble**.

ARIMA diestimasi lewat prosedur dua tahap **Hannan-Rissanen**: AR panjang untuk menaksir galat,
lalu regresi pada lag deret dan lag galat. Ini penaksir kuadrat terkecil bersyarat — bukan MLE
penuh seperti `statsmodels` — dan dipilih agar notebook tetap berjalan tanpa dependensi eksternal.
Bagian 18 membandingkannya dengan `statsmodels` bila tersedia.

In [ ]:
def f_naive(y, h, **kw): return np.repeat(y[-1], h)
def f_drift(y, h, **kw):
    s = (y[-1]-y[0])/(len(y)-1); return y[-1] + s*np.arange(1, h+1)
def f_mean_k(y, h, k=5, **kw): return np.repeat(np.mean(y[-k:]), h)

def _holt(y, al, be, h, phi=1.0):
    l, t = y[0], y[1]-y[0]
    for v in y[1:]:
        lp = l; l = al*v + (1-al)*(l+phi*t); t = be*(l-lp) + (1-be)*phi*t
    return np.array([l + sum(phi**(j+1) for j in range(i+1))*t for i in range(h)])

def f_holt(y, h, **kw):        return _holt(y, .6, .2, h, 1.0)
def f_holt_damped(y, h, **kw): return _holt(y, .6, .2, h, .85)
def f_holt_tuned(y, h, **kw):
    best, bp = np.inf, (.6,.2,1.)
    for al in [.2,.4,.6,.8]:
        for be in [.05,.1,.2,.3]:
            for phi in [.8,.9,.95,1.]:
                l, t = y[0], y[1]-y[0]; sse = 0
                for v in y[1:]:
                    sse += (v-(l+phi*t))**2
                    lp = l; l = al*v+(1-al)*(l+phi*t); t = be*(l-lp)+(1-be)*phi*t
                if sse < best: best, bp = sse, (al,be,phi)
    return _holt(y, bp[0], bp[1], h, bp[2])

def f_theta(y, h, **kw):
    n = len(y); x = np.arange(n)
    b = _ols(np.column_stack([np.ones(n), x]), y)
    lin = b[0] + b[1]*(n-1+np.arange(1, h+1))
    th2 = 2*y - (b[0] + b[1]*x)
    best, al = np.inf, .3
    for aa in np.arange(.1,.95,.05):
        l = th2[0]; sse = 0
        for v in th2[1:]:
            sse += (v-l)**2; l = aa*v + (1-aa)*l
        if sse < best: best, al = sse, aa
    l = th2[0]
    for v in th2[1:]: l = al*v + (1-al)*l
    return .5*np.repeat(l, h) + .5*lin

def _hannan_rissanen(z, p, q):
    n = len(z); L = max(8, p+q+2)
    if n-L < 10: return None
    X = np.column_stack([np.ones(n-L)] + [z[L-i-1:n-i-1] for i in range(L)])
    e = np.r_[np.zeros(L), z[L:] - X@_ols(X, z[L:])]
    m = max(p, q); rows = n-m
    if rows < p+q+3: return None
    cols = [np.ones(rows)] + [z[m-i-1:n-i-1] for i in range(p)] + [e[m-i-1:n-i-1] for i in range(q)]
    XX = np.column_stack(cols); yy = z[m:]
    b = _ols(XX, yy); res = yy - XX@b; k = XX.shape[1]
    aicc = rows*np.log((res@res)/rows) + 2*k + (2*k*(k+1))/max(rows-k-1, 1)
    return b, res, aicc, p, q, e

def f_arima(y, h, orde=None, kembalikan_orde=False, **kw):
    z = np.diff(y)
    kand = [orde] if orde else [(p,q) for p in range(3) for q in range(3)]
    best = None
    for (p,q) in kand:
        if p == 0 and q == 0: continue
        r = _hannan_rissanen(z, p, q)
        if r is None: continue
        if best is None or r[2] < best[2]: best = r
    if best is None:
        o = f_drift(y,h); return (o,(0,0)) if kembalikan_orde else o
    b, res, aicc, p, q, e = best
    zz, ee, lvl, out = list(z), list(e), y[-1], []
    for _ in range(h):
        v = b[0] + sum(b[1+i]*zz[-1-i] for i in range(p)) + sum(b[1+p+i]*ee[-1-i] for i in range(q))
        lvl += v; out.append(lvl); zz.append(v); ee.append(0.0)
    out = np.array(out)
    return (out, (p,q)) if kembalikan_orde else out

def f_arimax(y, h, exog=None, exog_depan=None, **kw):
    if exog is None: return f_arima(y, h)
    X = np.column_stack([np.ones(len(y)), exog]); b = _ols(X, y)
    return np.column_stack([np.ones(h), exog_depan])@b + f_arima(y - X@b, h)

def f_lin_tren(y, h, **kw):
    n = len(y); b = _ols(np.column_stack([np.ones(n), np.arange(n)]), y)
    return b[0] + b[1]*(n-1+np.arange(1, h+1))

def _fitur_lag(y, nlag):
    X, t = [], []
    for i in range(nlag, len(y)):
        X.append([y[i-j-1] for j in range(nlag)] + [i]); t.append(y[i])
    return np.array(X), np.array(t)

def f_ridge_lag(y, h, nlag=3, alpha=1.0, **kw):
    X, t = _fitur_lag(y, nlag); mu, sd = X.mean(0), X.std(0)+1e-9
    Xs = (X-mu)/sd
    b = np.linalg.solve(Xs.T@Xs + alpha*np.eye(Xs.shape[1]), Xs.T@(t-t.mean()))
    hist, out = list(y), []
    for _ in range(h):
        f = np.array([hist[-j-1] for j in range(nlag)] + [len(hist)])
        out.append(((f-mu)/sd)@b + t.mean()); hist.append(out[-1])
    return np.array(out)

def f_gbr_lag(y, h, nlag=3, **kw):
    X, t = _fitur_lag(y, nlag)
    if len(t) < 12: return f_drift(y, h)
    g = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=.08,
                                  random_state=RS).fit(X, t)
    hist, out = list(y), []
    for _ in range(h):
        out.append(g.predict(np.array([[hist[-j-1] for j in range(nlag)] + [len(hist)]]))[0])
        hist.append(out[-1])
    return np.array(out)

print('13 fungsi model siap.')

## 6. Walk-Forward Cross-Validation

Dua skema jendela diuji berdampingan, bukan hanya satu:

- **Expanding** — jendela latih tumbuh satu tahun tiap fold. Cocok bila proses stabil karena
  memanfaatkan seluruh sejarah.
- **Rolling** — jendela latih tetap 30 tahun dan bergeser. Lebih tahan bila struktur data
  berubah, karena sejarah jauh yang mungkin sudah tidak relevan dibuang.

Origin minimum 30 tahun (47% titik). Empat horizon diuji: **h=1, 3, 5, 10**, mencakup horizon
operasional sampai horizon perencanaan satu dekade. Semua model dilatih di ruang **log** lalu
dikembalikan lewat eksponensial, sehingga ramalan tidak pernah bernilai negatif dan galat
bersifat relatif.

In [ ]:
MODEL_FN = {
    'Naive'            : f_naive,
    'Drift'            : f_drift,
    'Rata-rata 5thn'   : f_mean_k,
    'Holt'             : f_holt,
    'Holt teredam'     : f_holt_damped,
    'Holt tuned'       : f_holt_tuned,
    'Theta'            : f_theta,
    'ARIMA (AICc)'     : f_arima,
    'Tren linear'      : f_lin_tren,
    'Ridge lag-3'      : f_ridge_lag,
    'GBR lag-3'        : f_gbr_lag,
}
EXOG_COLS = ['populasi_jt']
ORIGIN_MIN, HORIZONS, ROLL_WIN = 30, [1, 3, 5, 10], 30

def jalankan_cv(seri, skema='expanding', exog=None):
    y_full = np.log(seri.values.astype(float)); hasil = []
    for h in HORIZONS:
        for o in range(ORIGIN_MIN, len(y_full)-h+1):
            aw = 0 if skema == 'expanding' else max(0, o-ROLL_WIN)
            tr, akt = y_full[aw:o], seri.values[o:o+h]
            for nama, fn in MODEL_FN.items():
                try: p = np.exp(fn(tr, h))
                except Exception: continue
                if not np.all(np.isfinite(p)) or np.any(p <= 0): continue
                hasil.append({'skema':skema,'model':nama,'h':h,'origin':seri.index[o-1],
                              'mape':mape(akt,p),'rmse':rmse(akt,p),
                              'smape':smape(akt,p),'mase':mase(akt,p,np.exp(tr))})
            if exog is not None:
                try:
                    p = np.exp(f_arimax(tr, h, exog=exog[aw:o], exog_depan=exog[o:o+h]))
                    if np.all(np.isfinite(p)) and np.all(p > 0):
                        hasil.append({'skema':skema,'model':'ARIMAX (+populasi)','h':h,
                                      'origin':seri.index[o-1],'mape':mape(akt,p),'rmse':rmse(akt,p),
                                      'smape':smape(akt,p),'mase':mase(akt,p,np.exp(tr))})
                except Exception: pass
    return pd.DataFrame(hasil)

exog_pop = np.log(a['populasi_jt'].values)
cv_exp  = jalankan_cv(a[Y_COL], 'expanding', exog_pop)
cv_roll = jalankan_cv(a[Y_COL], 'rolling',   exog_pop)
cv = pd.concat([cv_exp, cv_roll], ignore_index=True)
print(f'Total evaluasi fold-model: {len(cv):,}')
print(f'Model dievaluasi         : {cv.model.nunique()}')
print(f'Fold per horizon (expanding): '
      f'{cv_exp[cv_exp.h==HORIZONS[0]].origin.nunique()} sampai '
      f'{cv_exp[cv_exp.h==HORIZONS[-1]].origin.nunique()}')

In [ ]:
ringkas = (cv.groupby(['skema','model','h'])[['mape','rmse','smape','mase']]
             .mean().round(3))
for sk in ['expanding','rolling']:
    print(f'\n{"="*70}\nSKEMA {sk.upper()}\n{"="*70}')
    for h in HORIZONS:
        t = ringkas.loc[sk].xs(h, level='h').sort_values('mape')
        print(f'\n--- h={h} tahun ---')
        display(t)
        print(f'Terbaik: {t.index[0]}  (MAPE {t.mape.iloc[0]:.3f}%, MASE {t.mase.iloc[0]:.3f})')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(19, 8))
for r, sk in enumerate(['expanding','rolling']):
    for c, h in enumerate(HORIZONS):
        ax = axes[r, c]
        t = ringkas.loc[sk].xs(h, level='h').sort_values('mape', ascending=True)
        w = [PALETTE['secondary'] if i == 0 else PALETTE['muted'] for i in range(len(t))]
        ax.barh(t.index[::-1], t['mape'][::-1], color=w[::-1])
        ax.set_title(f'{sk} — h={h}', fontsize=10)
        ax.set_xlabel('MAPE (%)')
        if c > 0: ax.set_yticklabels([])
        for i, v in enumerate(t['mape'][::-1]):
            ax.text(v+.02, i, f'{v:.2f}', va='center', fontsize=7)
fig.suptitle('MAPE Walk-Forward CV — 12 model x 4 horizon x 2 skema jendela',
             fontweight='bold', y=1.0)
plt.tight_layout()
fig.savefig(f'{SVG_DIR}/04_perbandingan_model_cv.svg', format='svg', bbox_inches='tight'); plt.show()

## 7. Kestabilan Antarfold

MAPE rata-rata dapat menyembunyikan model yang sesekali gagal parah. Sebaran per fold diperiksa
agar model terpilih terbukti stabil, bukan sekadar unggul di rata-rata.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))
for ax, h in zip(axes, HORIZONS):
    sub = cv[(cv.h == h) & (cv.skema == 'expanding')].dropna(subset=['mape'])
    urut = sub.groupby('model')['mape'].median().sort_values().index
    ax.boxplot([sub[sub.model == m]['mape'].values for m in urut], labels=urut, vert=False,
               patch_artist=True, boxprops=dict(facecolor=PALETTE['low'], alpha=.65),
               medianprops=dict(color=PALETTE['danger'], lw=2))
    ax.set_title(f'Sebaran MAPE antarfold — h={h}', fontsize=10); ax.set_xlabel('MAPE (%)')
    if ax is not axes[0]: ax.set_yticklabels([])
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/05_sebar_antarfold.svg', format='svg', bbox_inches='tight'); plt.show()

print('Rasio stabilitas (simpangan baku / rata-rata MAPE) — makin kecil makin stabil:\n')
for h in [3, 10]:
    sub = cv[(cv.h == h) & (cv.skema == 'expanding')].dropna(subset=['mape'])
    st = sub.groupby('model')['mape'].agg(['mean','std','max'])
    st['rasio'] = (st['std']/st['mean']).round(3)
    print(f'--- h={h} ---'); display(st.round(3).sort_values('rasio'))

## 7B. Model Juara Lintas Horizon, Metrik, dan Skema

Penentuan juara tidak memakai rata-rata MAPE mentah, karena skalanya berbeda antar horizon.
Dipakai **rata-rata peringkat** atas 4 horizon × 4 metrik × 2 skema jendela — total 32 peringkat
per model, jauh lebih ketat daripada agregasi dua horizon pada notebook bulanan.

In [ ]:
blok = []
for sk in ['expanding','rolling']:
    for h in HORIZONS:
        for met in ['mape','rmse','smape','mase']:
            blok.append(ringkas.loc[sk].xs(h, level='h')[met].rank())
peringkat = pd.concat(blok, axis=1)
peringkat.columns = [f'{sk[:3]}_h{h}_{m}' for sk in ['expanding','rolling']
                     for h in HORIZONS for m in ['mape','rmse','smape','mase']]
peringkat['rata_peringkat'] = peringkat.mean(axis=1)
peringkat = peringkat.sort_values('rata_peringkat')
display(peringkat[['rata_peringkat']].round(3))
BEST = peringkat.index[0]
print(f'Model juara lintas 32 kombinasi peringkat: "{BEST}"')
print(f'Runner-up: "{peringkat.index[1]}" (selisih peringkat '
      f'{peringkat.rata_peringkat.iloc[1]-peringkat.rata_peringkat.iloc[0]:.2f})')

In [ ]:
for h in [3, 10]:
    print(f'\n=== Skor MAPE per window — h={h}, skema expanding ===')
    t = (cv[(cv.h == h) & (cv.skema == 'expanding')]
         .pivot_table(index='origin', columns='model', values='mape').round(2))
    display(t.tail(12))

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5))
for ax, h in zip(axes, [3, 10]):
    sub = cv[(cv.h == h) & (cv.skema == 'expanding')].dropna(subset=['mape']).sort_values('origin')
    for nama in sub.model.unique():
        s = sub[sub.model == nama]; jagoan = (nama == BEST)
        ax.plot(s.origin, s.mape, lw=2.4 if jagoan else 1.0,
                color=PALETTE['danger'] if jagoan else PALETTE['muted'],
                alpha=1 if jagoan else .4, zorder=5 if jagoan else 2,
                label=f'{nama}  <- juara' if jagoan else None)
    ax.set_title(f'MAPE per window — h={h} tahun, semua model, "{BEST}" ditonjolkan', fontsize=10)
    ax.set_ylabel('MAPE (%)'); ax.set_xlabel('Tahun origin'); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/06_mape_per_window.svg', format='svg', bbox_inches='tight'); plt.show()

## 8. Uji Diebold-Mariano — Apakah Selisih Akurasi Bermakna?

Selisih MAPE antarmodel bisa saja kebetulan. Uji Diebold-Mariano menguji secara formal apakah
galat ramalan dua model berbeda secara statistik, dengan koreksi Harvey-Leybourne-Newbold untuk
horizon lebih dari satu. Tanpa uji ini, mengklaim satu model "lebih baik" hanya karena MAPE-nya
0,1 poin lebih rendah tidak dapat dipertahankan.

In [ ]:
def diebold_mariano(e1, e2, h=1, power=2):
    d = np.abs(np.asarray(e1))**power - np.abs(np.asarray(e2))**power
    n = len(d); dbar = d.mean(); g0 = np.var(d, ddof=0)
    gam = sum(2*(1-k/h)*np.mean((d[k:]-dbar)*(d[:-k]-dbar)) for k in range(1, h)) if h > 1 else 0
    v = (g0+gam)/n
    if v <= 0 or n < 5: return np.nan, np.nan
    stat = (dbar/np.sqrt(v)) * np.sqrt((n+1-2*h+h*(h-1)/n)/n)
    return stat, 2*(1-stats.t.cdf(abs(stat), n-1))

H_DM = 5
y_full = np.log(a[Y_COL].values)
galat = {}
for nama, fn in MODEL_FN.items():
    e = []
    for o in range(ORIGIN_MIN, len(y_full)-H_DM+1):
        try:
            p = np.exp(fn(y_full[:o], H_DM)); akt = a[Y_COL].values[o:o+H_DM]
            e.append(np.mean(akt-p))
        except Exception: e.append(np.nan)
    galat[nama] = np.array(e)

nama_model = list(galat)
P = pd.DataFrame(index=nama_model, columns=nama_model, dtype=float)
for i in nama_model:
    for j in nama_model:
        if i == j: P.loc[i,j] = np.nan; continue
        m_ = ~(np.isnan(galat[i]) | np.isnan(galat[j]))
        P.loc[i,j] = diebold_mariano(galat[i][m_], galat[j][m_], h=H_DM)[1]

plt.figure(figsize=(9.5, 7.5))
sns.heatmap(P.astype(float), annot=True, fmt='.3f', cmap='RdYlGn_r', vmin=0, vmax=.2,
            linewidths=.5, cbar_kws={'label':'p-value DM'})
plt.title(f'Uji Diebold-Mariano berpasangan (h={H_DM} tahun)\n'
          'hijau = selisih akurasi signifikan (p<0,05)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SVG_DIR}/07_diebold_mariano.svg', format='svg', bbox_inches='tight'); plt.show()

lawan = P.loc[BEST].dropna()
print(f'Juara "{BEST}" vs model lain (h={H_DM}):')
for m_, p_ in lawan.sort_values().items():
    print(f'   vs {m_:<20} p={p_:.4f}  {"berbeda signifikan" if p_ < .05 else "tidak berbeda signifikan"}')
print(f'\nModel yang TIDAK berbeda signifikan dari juara: '
      f'{[m for m,p_ in lawan.items() if p_ >= .05]}')
print('Konsekuensinya, ensemble atas model-model tersebut dapat dibenarkan: tidak ada bukti')
print('salah satunya lebih unggul, sehingga merata-ratakan mengurangi risiko salah pilih.')

## 9. Kalibrasi Selang Prediksi

Fan chart hanya berguna bila pitanya jujur. Di sini selang dibangun **empiris** dari sebaran
galat walk-forward per horizon, lalu cakupannya diperiksa: apakah pita nominal 80% benar-benar
memuat sekitar 80% kejadian? Pemeriksaan ini tidak ada pada notebook bulanan, padahal tanpa itu
lebar pita hanyalah asumsi.

In [ ]:
def galat_relatif_per_h(fn, y_log, aktual, H, origin_min):
    g = {h: [] for h in range(1, H+1)}
    for o in range(origin_min, len(y_log)-H+1):
        try: p = np.exp(fn(y_log[:o], H))
        except Exception: continue
        akt = aktual[o:o+H]
        for h in range(1, H+1): g[h].append((akt[h-1]-p[h-1])/p[h-1])
    return {h: np.array(v) for h, v in g.items()}

H_MAX = 11
FN_BEST = MODEL_FN[BEST]
gr = galat_relatif_per_h(FN_BEST, y_full, a[Y_COL].values, H_MAX, ORIGIN_MIN)

baris = []
for nominal, z in [(.50,.25), (.80,.10), (.95,.025)]:
    for h in [1, 3, 5, 10]:
        e = gr[h]
        lo, hi = np.quantile(e, z), np.quantile(e, 1-z)
        cakup = np.mean((e >= lo) & (e <= hi))
        baris.append({'nominal':f'{nominal:.0%}', 'h':h, 'cakupan empiris':f'{cakup:.0%}',
                      'batas bawah':round(lo,4), 'batas atas':round(hi,4), 'n fold':len(e)})
kal = pd.DataFrame(baris)
display(kal)

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for h, w in zip([1,3,5,10], [PALETTE['primary'],PALETTE['secondary'],PALETTE['accent'],PALETTE['danger']]):
    e = np.sort(gr[h]); q = np.arange(1, len(e)+1)/len(e)
    ax.plot(e*100, q, color=w, lw=1.8, label=f'h={h}')
ax.axvline(0, color='grey', lw=.8)
ax.set(xlabel='Galat relatif (%)', ylabel='Peluang kumulatif',
       title='Sebaran galat walk-forward per horizon — dasar pita fan chart')
ax.legend(); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/08_kalibrasi_selang.svg', format='svg', bbox_inches='tight'); plt.show()
print('Pita fan chart di Bagian 13 memakai kuantil empiris di atas, bukan asumsi normalitas.')

## 10. Robustness: Patahan Struktural

Notebook bulanan menguji satu patahan yang sudah diketahui (onset COVID-19). Di sini titik
patahannya **tidak diketahui di muka**, sehingga dipakai dua prosedur: **CUSUM** atas residual
rekursif untuk mendeteksi kapan hubungan bergeser, dan **uji Chow** yang dijalankan pada setiap
tahun kandidat untuk menemukan patahan paling tajam. Sesudahnya, manfaat memasukkan dummy
patahan diuji lewat CV — bukan diasumsikan menolong.

In [ ]:
def cusum_rekursif(y):
    n = len(y); x = np.arange(n); k = 2; w = []
    for t in range(k, n):
        X = np.column_stack([np.ones(t), x[:t]]); b = _ols(X, y[:t])
        f = y[t] - np.array([1, x[t]])@b
        X2 = np.column_stack([np.ones(t), x[:t]])
        d = 1 + np.array([1, x[t]])@np.linalg.pinv(X2.T@X2)@np.array([1, x[t]])
        w.append(f/np.sqrt(d))
    w = np.array(w); s = w.std(ddof=1)
    return np.cumsum(w)/s, n, k

W, n_, k_ = cusum_rekursif(ly)
tt = np.arange(k_+1, n_+1)
batas = 0.948*np.sqrt(n_-k_)*(1 + 2*(tt-k_)/(n_-k_))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(a.index[k_:], W, color=PALETTE['primary'], lw=1.9, label='CUSUM')
ax.plot(a.index[k_:], batas, color=PALETTE['danger'], ls='--', lw=1, label='Batas 5%')
ax.plot(a.index[k_:], -batas, color=PALETTE['danger'], ls='--', lw=1)
ax.axhline(0, color='grey', lw=.8)
ax.set(title='Uji CUSUM atas residual rekursif — deteksi pergeseran struktural',
       xlabel='Tahun', ylabel='CUSUM terstandardisasi')
ax.legend(); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/09_cusum.svg', format='svg', bbox_inches='tight'); plt.show()
lewat = a.index[k_:][np.abs(W) > batas]
print('Tahun saat CUSUM melewati batas 5%:', list(lewat) if len(lewat) else 'tidak ada')

In [ ]:
def chow(y, brk):
    n = len(y); x = np.arange(n)
    X = np.column_stack([np.ones(n), x]); e = y - X@_ols(X, y); S = e@e
    X1, y1 = X[:brk], y[:brk]; X2, y2 = X[brk:], y[brk:]
    if len(y1) < 5 or len(y2) < 5: return np.nan, np.nan
    e1 = y1 - X1@_ols(X1, y1); e2 = y2 - X2@_ols(X2, y2)
    S1, S2 = e1@e1, e2@e2; k = 2
    F = ((S-(S1+S2))/k) / ((S1+S2)/(n-2*k))
    return F, 1-stats.f.cdf(F, k, n-2*k)

kand = [(a.index[i], *chow(ly, i)) for i in range(8, len(ly)-8)]
ch = pd.DataFrame(kand, columns=['tahun','F','p']).dropna()
top = ch.nlargest(5, 'F')
print('Lima kandidat patahan struktural dengan statistik Chow tertinggi:')
display(top.round(4))
BRK = int(top.iloc[0].tahun)
print(f'Patahan terkuat: {BRK} (F={top.iloc[0].F:.2f}, p={top.iloc[0].p:.5f})')

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(ch.tahun, ch.F, color=PALETTE['primary'], lw=1.8)
ax.axvline(BRK, color=PALETTE['danger'], ls='--', lw=1.4, label=f'patahan terkuat: {BRK}')
ax.set(title='Statistik Chow pada setiap tahun kandidat patahan', xlabel='Tahun', ylabel='F')
ax.legend(); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/10_chow.svg', format='svg', bbox_inches='tight'); plt.show()

In [ ]:
t_arr = np.arange(len(a)); idx_brk = list(a.index).index(BRK)
SKEMA_DUMMY = {
    'Tanpa dummy'          : None,
    'Step (0/1 sejak patahan)': (t_arr >= idx_brk).astype(float),
    'Waktu-sejak-patahan'  : np.clip(t_arr-idx_brk, 0, None).astype(float),
}

def f_dummy(y, h, dummy_penuh=None, i0=0, **kw):
    if dummy_penuh is None: return f_arima(y, h)
    n = len(y)
    return f_arimax(y, h, exog=dummy_penuh[i0:i0+n], exog_depan=dummy_penuh[i0+n:i0+n+h])

baris = []
for nm, dm in SKEMA_DUMMY.items():
    for h in [3, 5, 10]:
        errs = []
        for o in range(ORIGIN_MIN, len(y_full)-h+1):
            try:
                p = np.exp(f_dummy(y_full[:o], h, dm, 0))
                if np.all(np.isfinite(p)) and np.all(p > 0):
                    errs.append(mape(a[Y_COL].values[o:o+h], p))
            except Exception: pass
        baris.append({'skema':nm, 'h':h, 'MAPE':np.mean(errs) if errs else np.nan})
rb = pd.DataFrame(baris).pivot(index='skema', columns='h', values='MAPE').round(3)
rb.columns = [f'h={c}' for c in rb.columns]
display(rb)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3); lb = .25
for i, sk in enumerate(SKEMA_DUMMY):
    ax.bar(x+i*lb, rb.loc[sk].values, lb, label=sk,
           color=[PALETTE['muted'], PALETTE['accent'], PALETTE['secondary']][i])
ax.set_xticks(x+lb); ax.set_xticklabels(['h=3','h=5','h=10'])
ax.set(ylabel='MAPE rata-rata (%)', title=f'Manfaat regressor patahan struktural ({BRK}) pada akurasi')
ax.legend(fontsize=8.5); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/11_robustness_patahan.svg', format='svg', bbox_inches='tight'); plt.show()
best_sk = rb.mean(axis=1).idxmin()
print(f'Skema dengan MAPE rata-rata terendah: "{best_sk}"')
print('Bila "Tanpa dummy" yang menang, patahan itu nyata secara historis tetapi TIDAK membantu')
print('meramal ke depan — temuan negatif yang tetap layak dilaporkan.')

## 11. Model Final dan Diagnostik Residual

Model juara dilatih ulang pada seluruh 64 tahun. Residualnya diuji lewat empat prosedur:
Ljung-Box (autokorelasi tersisa), ARCH-LM (heteroskedastisitas bersyarat — tidak diuji pada
notebook bulanan, padahal bila ada, pita selang menjadi salah lebar), Jarque-Bera (normalitas),
dan QQ-plot.

In [ ]:
def acf_v(y, nlags=20):
    y = np.asarray(y,float) - np.mean(y); d = y@y
    return np.array([1.0] + [(y[k:]@y[:-k])/d for k in range(1, nlags+1)])

def ljung_box(e, lags):
    e = np.asarray(e,float); n = len(e); r = acf_v(e, max(lags)); out = []
    for L in lags:
        Q = n*(n+2)*np.sum([r[k]**2/(n-k) for k in range(1, L+1)])
        out.append({'lag':L, 'stat':round(Q,3), 'p':round(1-stats.chi2.cdf(Q, L), 4)})
    return pd.DataFrame(out)

def arch_lm(e, lags=4):
    e2 = np.asarray(e,float)**2; n = len(e2)-lags
    X = np.column_stack([np.ones(n)] + [e2[lags-i-1:-i-1] for i in range(lags)])
    yy = e2[lags:]; res = yy - X@_ols(X, yy)
    r2 = 1 - (res@res)/(((yy-yy.mean())**2).sum())
    return n*r2, 1-stats.chi2.cdf(n*r2, lags)

# residual in-sample model juara: ramalan satu langkah berulang
resid = []
for o in range(ORIGIN_MIN, len(y_full)):
    try: resid.append(y_full[o] - FN_BEST(y_full[:o], 1)[0])
    except Exception: resid.append(np.nan)
resid = np.array(resid); resid = resid[np.isfinite(resid)]
thn_res = a.index[ORIGIN_MIN:ORIGIN_MIN+len(resid)]

print(f'Model final: "{BEST}" dilatih pada {len(a)} tahun penuh')
_, orde = f_arima(y_full, 1, kembalikan_orde=True)
print(f'Orde ARIMA pembanding terpilih via AICc: (p,d,q) = ({orde[0]}, {D_ORDER}, {orde[1]})\n')
print('Uji Ljung-Box (H0: tidak ada autokorelasi tersisa):'); display(ljung_box(resid, [5, 10]))
st_a, p_a = arch_lm(resid); print(f'Uji ARCH-LM(4): stat={st_a:.3f}, p={p_a:.4f} '
      f'({"ADA efek ARCH" if p_a < .05 else "tidak ada efek ARCH"})')
jb = stats.jarque_bera(resid); print(f'Uji Jarque-Bera: stat={jb[0]:.3f}, p={jb[1]:.4f} '
      f'({"tidak normal" if jb[1] < .05 else "konsisten dengan normal"})')

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
axes[0].plot(thn_res, resid, color=PALETTE['primary'], lw=1.2, marker='o', ms=3)
axes[0].axhline(0, color='grey', lw=.8); axes[0].set_title('Residual sepanjang waktu')
rr = acf_v(resid, 15); bnd = 1.96/np.sqrt(len(resid))
axes[1].bar(range(16), rr, color=PALETTE['primary'], width=.35)
axes[1].axhline(bnd, color=PALETTE['danger'], ls='--', lw=.9)
axes[1].axhline(-bnd, color=PALETTE['danger'], ls='--', lw=.9)
axes[1].axhline(0, color='grey', lw=.8); axes[1].set_title('ACF residual')
stats.probplot(resid, dist='norm', plot=axes[2]); axes[2].set_title('QQ-plot residual')
axes[3].hist(resid, bins=14, color=PALETTE['low'], edgecolor=PALETTE['dark'])
axes[3].set_title('Histogram residual')
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/12_diagnostik_residual.svg', format='svg', bbox_inches='tight'); plt.show()

## 12. Rekonsiliasi Hirarkis

ASEAN adalah jumlah sepuluh negara, sehingga tersedia dua jalur ramalan yang harus konsisten:
meramal agregatnya langsung (**top-down**), atau meramal tiap negara lalu menjumlahkannya
(**bottom-up**). Keduanya jarang sama persis. Bagian ini membandingkan akurasi keduanya lewat
walk-forward CV, lalu menyusun kombinasi berbobot optimal — pendekatan yang tidak ada pada
notebook bulanan karena datanya memang tidak berhierarki.

In [ ]:
NEGARA = list(neg.columns)
def cv_hirarki(H):
    lang, bawah, aktual = [], [], []
    for o in range(ORIGIN_MIN, len(a)-H+1):
        akt = a[Y_COL].values[o:o+H].sum()
        p_lang = np.exp(FN_BEST(np.log(a[Y_COL].values[:o]), H)).sum()
        p_bawah = 0
        for c in NEGARA:
            yc = np.log(neg[c].values[:o])
            try: p_bawah += np.exp(FN_BEST(yc, H)).sum()
            except Exception: p_bawah = np.nan; break
        if np.isfinite(p_bawah):
            lang.append(p_lang); bawah.append(p_bawah); aktual.append(akt)
    return np.array(lang), np.array(bawah), np.array(aktual)

baris = []
simpan = {}
for H in [3, 5, 10]:
    l_, b_, k_ = cv_hirarki(H)
    simpan[H] = (l_, b_, k_)
    A = np.column_stack([l_, b_])
    w = _ols(A, k_); w = np.clip(w, 0, None); w = w/w.sum()
    komb = A@w
    baris.append({'h':H, 'MAPE langsung':mape(k_, l_), 'MAPE bottom-up':mape(k_, b_),
                  'MAPE kombinasi':mape(k_, komb),
                  'bobot langsung':round(w[0],3), 'bobot bottom-up':round(w[1],3)})
hir = pd.DataFrame(baris).round(3); display(hir)

W_OPT = hir.loc[hir.h == 5, ['bobot langsung','bobot bottom-up']].values[0]
print(f'Bobot rekonsiliasi dipakai (dari h=5): langsung {W_OPT[0]:.3f}, bottom-up {W_OPT[1]:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, H in zip(axes, [3, 5, 10]):
    l_, b_, k_ = simpan[H]
    ax.plot(k_, color=PALETTE['dark'], lw=2, label='Aktual')
    ax.plot(l_, color=PALETTE['danger'], lw=1.5, ls='--', label='Langsung (top-down)')
    ax.plot(b_, color=PALETTE['secondary'], lw=1.5, ls=':', label='Bottom-up (10 negara)')
    ax.set_title(f'Jumlah {H} tahun ke depan — h={H}', fontsize=10)
    ax.set_xlabel('Fold'); ax.legend(fontsize=7.5)
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/13_rekonsiliasi_hirarkis.svg', format='svg', bbox_inches='tight'); plt.show()

## 13. Proyeksi Akhir — Fan Chart

Titik ramalan memakai **ensemble** atas model-model yang pada Bagian 8 terbukti tidak berbeda
signifikan dari juara (merata-ratakan mengurangi risiko salah pilih ketika bukti tidak memihak
satu model). Pita ketidakpastian memakai **kuantil galat empiris** per horizon dari Bagian 9,
bukan asumsi normalitas — sehingga lebar pita mencerminkan seberapa sering model ini benar-benar
meleset di masa lalu.

In [ ]:
H_FINAL = 11
tidak_beda = [m for m, p_ in P.loc[BEST].dropna().items() if p_ >= .05]
ANGGOTA = [BEST] + [m for m in tidak_beda if m in MODEL_FN]
ANGGOTA = [m for m in ANGGOTA if m not in ('Naive','Rata-rata 5thn')][:4]
print('Anggota ensemble:', ANGGOTA)

pred = {m: np.exp(MODEL_FN[m](y_full, H_FINAL)) for m in ANGGOTA}
titik = np.mean(list(pred.values()), axis=0)
tahun_depan = np.arange(a.index[-1]+1, a.index[-1]+1+H_FINAL)

q = {}
for lv, z in [('50',.25), ('80',.10), ('95',.025)]:
    lo = np.array([np.quantile(gr[min(h, H_MAX)], z)     for h in range(1, H_FINAL+1)])
    hi = np.array([np.quantile(gr[min(h, H_MAX)], 1-z)   for h in range(1, H_FINAL+1)])
    q[lv] = (titik*(1+lo), titik*(1+hi))

proyeksi = pd.DataFrame({'tahun':tahun_depan, 'ensemble':titik,
                         **{f'{m}':pred[m] for m in ANGGOTA},
                         'ci50_bawah':q['50'][0], 'ci50_atas':q['50'][1],
                         'ci80_bawah':q['80'][0], 'ci80_atas':q['80'][1],
                         'ci95_bawah':q['95'][0], 'ci95_atas':q['95'][1]}).set_index('tahun')
display(proyeksi.round(2))
proyeksi.round(4).to_csv('proyeksi_metana_asean.csv')

In [ ]:
def fan_chart(h_tampil, judul, berkas):
    p = proyeksi.iloc[:h_tampil]
    fig, ax = plt.subplots(figsize=(11.5, 5.6))
    hist = a[Y_COL].iloc[-30:]
    ax.plot(hist.index, hist.values, color=PALETTE['dark'], lw=2, label='Aktual 1995-2024')
    for lv, al, w in [('95',.18,PALETTE['low']), ('80',.30,PALETTE['low']), ('50',.45,PALETTE['low'])]:
        ax.fill_between(p.index, p[f'ci{lv}_bawah'], p[f'ci{lv}_atas'],
                        color=w, alpha=al, label=f'Selang {lv}%')
    ax.plot(np.r_[hist.index[-1], p.index], np.r_[hist.values[-1], p['ensemble']],
            color=PALETTE['danger'], lw=2.2, ls='--', label='Ensemble')
    ax.axvline(a.index[-1], color=PALETTE['muted'], ls=':', lw=1)
    for t_ in [2030, 2035]:
        if t_ in p.index:
            v = p.loc[t_,'ensemble']
            ax.annotate(f'{t_}: {v:,.0f}\n({100*(v-hist.values[-1])/hist.values[-1]:+.1f}%)',
                        (t_, v), fontsize=8.5, weight='bold', color=PALETTE['danger'],
                        xytext=(6, 8), textcoords='offset points')
    ax.set(title=judul, xlabel='Tahun', ylabel='Metana ASEAN (Mt CO2e/tahun)')
    ax.legend(fontsize=8, loc='upper left'); plt.tight_layout()
    fig.savefig(f'{SVG_DIR}/{berkas}', format='svg', bbox_inches='tight'); plt.show()

fan_chart(3,  'Proyeksi Metana ASEAN — horizon 3 tahun (2025-2027)',  '14_fan_h3.svg')
fan_chart(6,  'Proyeksi Metana ASEAN — horizon 6 tahun (2025-2030)',  '15_fan_h6.svg')
fan_chart(11, 'Proyeksi Metana ASEAN — horizon 11 tahun (2025-2035)', '16_fan_h11.svg')

## 13B. Tuning Lanjutan — 266 Konfigurasi dengan Validasi Tiga Arah

Bagian 6-7 memilih model dari 12 kandidat berparameter tetap. Di sini ruang pencarian diperluas
menjadi **266 konfigurasi** dengan hiperparameter yang divariasikan: panjang jendela drift,
faktor peredaman, drift kokoh berbasis median, grid Holt (alpha x beta x phi), bobot Theta,
orde ARIMA sampai (3,2,3), serta strategi **direct multi-step** yang melatih regresi terpisah
untuk tiap horizon alih-alih mengiterasi ramalan satu langkah.

Masalahnya, memilih pemenang dari 266 kandidat pada data yang sama akan **overfit terhadap
pemilihan itu sendiri**. Karena itu dipakai pembagian tiga arah:

| Bagian | Origin | Tahun | Fungsi |
|---|---|---|---|
| INNER | 30–41 | 1991–2002 | memilih hiperparameter tiap keluarga |
| MID | 42–52 | 2003–2013 | memilih *strategi* (seleksi tunggal vs ensemble) |
| OUTER | 53–63 | 2014–2024 | menilai — tidak pernah disentuh kedua langkah di atas |


In [ ]:
def drift_jendela(y, h, k=None, phi=1.0):
    yy = y if (k is None or k >= len(y)) else y[-k:]
    s = (yy[-1]-yy[0])/(len(yy)-1)
    return y[-1] + s*np.array([sum(phi**(j+1) for j in range(i+1)) for i in range(h)])

def drift_median(y, h, k=None, phi=1.0):
    yy = y if (k is None or k >= len(y)) else y[-k:]
    s = np.median(np.diff(yy))
    return y[-1] + s*np.array([sum(phi**(j+1) for j in range(i+1)) for i in range(h)])

def holt_par(y, h, al, be, phi):
    l, t = y[0], y[1]-y[0]
    for v in y[1:]:
        lp = l; l = al*v + (1-al)*(l+phi*t); t = be*(l-lp) + (1-be)*phi*t
    return np.array([l + sum(phi**(j+1) for j in range(i+1))*t for i in range(h)])

def theta_par(y, h, w, al):
    n = len(y); x = np.arange(n)
    b = _ols(np.column_stack([np.ones(n), x]), y)
    lin = b[0] + b[1]*(n-1+np.arange(1, h+1)); th2 = 2*y - (b[0] + b[1]*x)
    l = th2[0]
    for v in th2[1:]: l = al*v + (1-al)*l
    return w*np.repeat(l, h) + (1-w)*lin

def arima_par(y, h, p, q, d=1):
    z = np.diff(y, n=d); r = _hannan_rissanen(z, p, q)
    if r is None: return drift_jendela(y, h)
    b, res, aicc, p, q, e = r
    zz, ee, out_z = list(z), list(e), []
    for _ in range(h):
        v = b[0] + sum(b[1+i]*zz[-1-i] for i in range(p)) + sum(b[1+p+i]*ee[-1-i] for i in range(q))
        zz.append(v); ee.append(0.0); out_z.append(v)
    out = np.array(out_z)
    for _ in range(d):
        base = y[-1] if d == 1 else np.diff(y, n=d-1)[-1]
        out = base + np.cumsum(out)
    return out

def lin_tren_jendela(y, h, k=None):
    yy = y if (k is None or k >= len(y)) else y[-k:]
    n = len(yy); b = _ols(np.column_stack([np.ones(n), np.arange(n)]), yy)
    return b[0] + b[1]*(n-1+np.arange(1, h+1))

def direct_multistep(y, h, k=None):
    """Satu regresi khusus per horizon, bukan iterasi rekursif."""
    yy = y if (k is None or k >= len(y)) else y[-k:]
    n = len(yy); out = []
    for step in range(1, h+1):
        X, t = [], []
        for i in range(2, n-step):
            X.append([1, yy[i], yy[i]-yy[i-1], i]); t.append(yy[i+step])
        if len(t) < 8: out.append(yy[-1]); continue
        b = _ols(np.array(X), np.array(t))
        out.append(np.array([1, yy[-1], yy[-1]-yy[-2], n-1])@b)
    return np.array(out)

RUANG = {}
for k in [8,12,16,20,25,30,40,None]:
    for phi in [0.90,0.95,0.98,1.00]:
        RUANG[f'DriftJendela(k={k},phi={phi})'] = lambda y,h,k=k,phi=phi: drift_jendela(y,h,k,phi)
        RUANG[f'DriftMedian(k={k},phi={phi})']  = lambda y,h,k=k,phi=phi: drift_median(y,h,k,phi)
for al in [0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    for be in [0.02,0.05,0.1,0.15,0.2,0.3]:
        for phi in [0.92,0.96,1.0]:
            RUANG[f'Holt(a={al},b={be},phi={phi})'] = lambda y,h,al=al,be=be,phi=phi: holt_par(y,h,al,be,phi)
for w in [0.2,0.35,0.5,0.65,0.8]:
    for al in [0.2,0.4,0.6,0.8]:
        RUANG[f'Theta(w={w},a={al})'] = lambda y,h,w=w,al=al: theta_par(y,h,w,al)
for p in range(4):
    for q in range(4):
        if p == 0 and q == 0: continue
        for d in [1,2]:
            RUANG[f'ARIMA(p={p},d={d},q={q})'] = lambda y,h,p=p,q=q,d=d: arima_par(y,h,p,q,d)
for k in [12,20,30,None]:
    RUANG[f'LinTren(k={k})'] = lambda y,h,k=k: lin_tren_jendela(y,h,k)
    RUANG[f'Direct(k={k})']  = lambda y,h,k=k: direct_multistep(y,h,k)

print(f'Kandidat konfigurasi dalam ruang pencarian: {len(RUANG)}')

In [ ]:
INNER, MID, OUTER = (30,42), (42,53), (53,64)
NN = len(y_full); DRIFT_POLOS = lambda y,h: drift_jendela(y,h,None,1.0)

def _p(fn,h,o):
    try:
        pp = np.exp(fn(y_full[:o],h))
        return pp if (np.all(np.isfinite(pp)) and np.all(pp>0)) else None
    except Exception: return None

def skor_ens(fns,h,rng):
    o0,o1 = rng; e=[]
    for o in range(o0, min(o1, NN-h+1)):
        ps=[_p(f,h,o) for f in fns]
        if any(x is None for x in ps): continue
        e.append(mape(a[Y_COL].values[o:o+h], np.array(ps).mean(0)))
    return np.mean(e) if e else np.nan

STRATEGI = ['Drift polos','top-1','ens top-3','ens top-5','ens keluarga']
bahan, mid_tab = {}, []
for h in HORIZONS:
    st = {nm: skor_ens([fn],h,INNER) for nm,fn in RUANG.items()}
    st = {k_:v for k_,v in st.items() if np.isfinite(v)}
    urut = sorted(st, key=st.get); fam = {}
    for u in urut:
        f = u.split('(')[0]
        if f not in fam: fam[f] = u
    paket = {'Drift polos':[DRIFT_POLOS], 'top-1':[RUANG[urut[0]]],
             'ens top-3':[RUANG[u] for u in urut[:3]], 'ens top-5':[RUANG[u] for u in urut[:5]],
             'ens keluarga':[RUANG[u] for u in fam.values()]}
    bahan[h] = (paket, urut, fam)
    mid_tab.append({'h':h, **{s: skor_ens(paket[s],h,MID) for s in STRATEGI}})
mid_df = pd.DataFrame(mid_tab).set_index('h').round(3)
print('Skor pada fold MID (hiperparameter berasal dari INNER):'); display(mid_df)

In [ ]:
hasil_outer = []
for h in HORIZONS:
    paket,_,_ = bahan[h]
    pilih = mid_df.loc[h].idxmin()
    ml = skor_ens(paket['Drift polos'],h,OUTER); mb = skor_ens(paket[pilih],h,OUTER)
    hasil_outer.append({'h':h, 'strategi terpilih (dari MID)':pilih,
                        'OUTER Drift polos':ml, 'OUTER terpilih':mb,
                        'perbaikan %':100*(mb-ml)/ml})
out_df = pd.DataFrame(hasil_outer).set_index('h').round(3)
display(out_df)
print(f"Rata-rata perbaikan pada fold OUTER: "
      f"{100*(out_df['OUTER terpilih'].mean()-out_df['OUTER Drift polos'].mean())/out_df['OUTER Drift polos'].mean():+.1f}%")
print()
print('Tafsir: pada horizon pendek Drift polos tidak terkalahkan -- tuning justru menambah ragam.')
print('Pada horizon panjang ensemble konfigurasi tertuning menang telak, karena di sana penaksiran')
print('tren mendominasi galat dan merata-ratakan beberapa penaksir tren menurunkan ragamnya.')
print()
print('ATURAN YANG DIPAKAI: h <= 3 -> Drift polos ; h >= 4 -> ensemble top-5 tertuning.')

fig, ax = plt.subplots(figsize=(9, 4.4))
x = np.arange(len(HORIZONS)); lb = .38
ax.bar(x, out_df['OUTER Drift polos'], lb, label='Drift polos', color=PALETTE['muted'])
ax.bar(x+lb, out_df['OUTER terpilih'], lb, label='Strategi terpilih', color=PALETTE['secondary'])
ax.set_xticks(x+lb/2); ax.set_xticklabels([f'h={h}' for h in HORIZONS])
ax.set(ylabel='MAPE fold OUTER (%)', title='Hasil tuning pada fold yang belum pernah dilihat')
ax.legend(fontsize=8.5); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/20_hasil_tuning.svg', format='svg', bbox_inches='tight'); plt.show()

## 13C. Proyeksi Final Tertuning dan Fan Chart Rentang Penuh

Aturan per-horizon yang tervalidasi di atas diterapkan: untuk tiap h dari 1 sampai 11,
hiperparameter dipilih ulang memakai seluruh 64 tahun, lalu ramalannya dirata-ratakan sesuai
strategi yang berlaku bagi horizon itu. Selang prediksi tetap memakai kuantil galat empiris —
dihitung ulang untuk komposisi ensemble yang baru.

Fan chart ditampilkan dari **titik awal data (1961)**, bukan hanya tiga dekade terakhir,
agar pembaca melihat proyeksi dalam konteks seluruh sejarah seri.

In [ ]:
ATURAN = {h: ('Drift polos' if h <= 3 else 'ens top-5') for h in range(1, 12)}
fn_h, komposisi_h, cv_h = {}, {}, {}
for h in range(1, 12):
    if ATURAN[h] == 'Drift polos':
        fn_h[h] = [DRIFT_POLOS]; komposisi_h[h] = ['Drift polos']
    else:
        st = {nm: skor_ens([fn],h,(ORIGIN_MIN,NN)) for nm,fn in RUANG.items()}
        st = {k_:v for k_,v in st.items() if np.isfinite(v)}
        urut = sorted(st, key=st.get)[:5]
        fn_h[h] = [RUANG[u] for u in urut]; komposisi_h[h] = urut
    cv_h[h] = skor_ens(fn_h[h], h, (ORIGIN_MIN,NN))
display(pd.DataFrame([{'h':h, 'strategi':ATURAN[h], 'MAPE CV (%)':round(cv_h[h],3),
                       'anggota':', '.join(komposisi_h[h][:3]) + (' ...' if len(komposisi_h[h])>3 else '')}
                      for h in range(1,12)]).set_index('h'))

titik_t = np.array([np.mean([np.exp(f(y_full, h))[-1] for f in fn_h[h]]) for h in range(1,12)])
tahun_t = np.arange(a.index[-1]+1, a.index[-1]+12)

gal_t = {}
for h in range(1, 12):
    e = []
    for o in range(ORIGIN_MIN, NN-h+1):
        ps = [_p(f,h,o) for f in fn_h[h]]
        if any(x is None for x in ps): continue
        g = np.array(ps).mean(0)
        e.append((a[Y_COL].values[o+h-1]-g[h-1])/g[h-1])
    gal_t[h] = np.array(e)

qq = {}
for lv, z in [('50',.25), ('80',.10), ('95',.025)]:
    qq[lv] = (titik_t*(1+np.array([np.quantile(gal_t[h], z)   for h in range(1,12)])),
              titik_t*(1+np.array([np.quantile(gal_t[h], 1-z) for h in range(1,12)])))

proy_t = pd.DataFrame({'tahun':tahun_t, 'ramalan':titik_t,
                       'ci50_bawah':qq['50'][0],'ci50_atas':qq['50'][1],
                       'ci80_bawah':qq['80'][0],'ci80_atas':qq['80'][1],
                       'ci95_bawah':qq['95'][0],'ci95_atas':qq['95'][1]}).set_index('tahun')
display(proy_t.round(1))
proy_t.round(4).to_csv('proyeksi_metana_asean_tuned.csv')
for t_ in [2030, 2035]:
    v = proy_t.loc[t_,'ramalan']
    print(f'{t_}: {v:,.1f} Mt ({100*(v-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}% vs 2024) | '
          f'selang 80% {proy_t.loc[t_,"ci80_bawah"]:,.1f} - {proy_t.loc[t_,"ci80_atas"]:,.1f}')

In [ ]:
def fan_penuh(mulai, judul, berkas, log=False):
    p = proy_t
    fig, ax = plt.subplots(figsize=(13, 6))
    hist = a[Y_COL][a.index >= mulai]
    ax.plot(hist.index, hist.values, color=PALETTE['dark'], lw=2, label=f'Aktual {mulai}-2024')
    for lv, al in [('95',.16), ('80',.28), ('50',.42)]:
        ax.fill_between(p.index, p[f'ci{lv}_bawah'], p[f'ci{lv}_atas'],
                        color=PALETTE['low'], alpha=al, label=f'Selang {lv}%')
    ax.plot(np.r_[hist.index[-1], p.index], np.r_[hist.values[-1], p['ramalan']],
            color=PALETTE['danger'], lw=2.4, ls='--', label='Ramalan ensemble tertuning')
    ax.axvline(a.index[-1], color=PALETTE['muted'], ls=':', lw=1.1)
    ax.annotate('data berakhir\n2024', (a.index[-1], hist.values.min()), fontsize=7.5,
                color=PALETTE['muted'], xytext=(-38, 6), textcoords='offset points')
    for t_ in [2030, 2035]:
        v = p.loc[t_,'ramalan']
        ax.annotate(f'{t_}: {v:,.0f} Mt\n({100*(v-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}%)',
                    (t_, v), fontsize=9, weight='bold', color=PALETTE['danger'],
                    xytext=(8, 10), textcoords='offset points')
    if log: ax.set_yscale('log')
    ax.set(title=judul, xlabel='Tahun', ylabel='Metana ASEAN (Mt CO2e/tahun)')
    ax.legend(fontsize=8.5, loc='upper left'); plt.tight_layout()
    fig.savefig(f'{SVG_DIR}/{berkas}', format='svg', bbox_inches='tight'); plt.show()

fan_penuh(1961, 'Metana ASEAN 1961-2035 — seluruh rentang data dan proyeksinya', '21_fan_penuh.svg')
fan_penuh(1961, 'Metana ASEAN 1961-2035 (skala log) — laju pertumbuhan melandai',
          '22_fan_penuh_log.svg', log=True)
fan_penuh(1995, 'Metana ASEAN 1995-2035 — perbesaran tiga dekade terakhir', '23_fan_zoom.svg')

lama_t = np.exp(DRIFT_POLOS(y_full, 11))
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.plot(a.index, a[Y_COL].values, color=PALETTE['dark'], lw=1.9, label='Aktual 1961-2024')
ax.plot(np.r_[a.index[-1], tahun_t], np.r_[a[Y_COL].iloc[-1], lama_t], color=PALETTE['muted'],
        lw=1.8, ls=':', label=f'Drift polos (2035: {lama_t[-1]:,.0f} Mt)')
ax.plot(np.r_[a.index[-1], tahun_t], np.r_[a[Y_COL].iloc[-1], titik_t], color=PALETTE['danger'],
        lw=2.3, ls='--', label=f'Ensemble tertuning (2035: {titik_t[-1]:,.0f} Mt)')
ax.axvline(a.index[-1], color=PALETTE['muted'], ls=':', lw=1)
ax.set(title='Dampak tuning terhadap lintasan proyeksi', xlabel='Tahun',
       ylabel='Metana ASEAN (Mt CO2e/tahun)')
ax.legend(fontsize=8.5); plt.tight_layout()
fig.savefig(f'{SVG_DIR}/24_lama_vs_tuned.svg', format='svg', bbox_inches='tight'); plt.show()

## 14. Seri Sekunder

Tiga seri pendukung panel pembuka diramal dengan pipeline yang sama: pemilihan model lewat
walk-forward CV tersendiri, bukan mewarisi juara dari seri metana.

In [ ]:
SEKUNDER = {'n2o':'N2O ASEAN (Mt CO2e)', 'pangsa_ch4':'Pangsa metana global (%)',
            'kum_luc':'Kumulatif CO2 alih guna lahan (Mt)', 'populasi_jt':'Populasi ASEAN (juta)'}
ring_sek, proy_sek = [], {}
for kol, lab in SEKUNDER.items():
    y_s = np.log(a[kol].values.astype(float))
    skor = {}
    for nm, fn in MODEL_FN.items():
        e = []
        for o in range(ORIGIN_MIN, len(y_s)-5+1):
            try:
                p = np.exp(fn(y_s[:o], 5))
                if np.all(np.isfinite(p)) and np.all(p > 0):
                    e.append(mape(a[kol].values[o:o+5], p))
            except Exception: pass
        if e: skor[nm] = np.mean(e)
    juara = min(skor, key=skor.get)
    pk = np.exp(MODEL_FN[juara](y_s, H_FINAL))
    proy_sek[kol] = pk
    ring_sek.append({'seri':lab, 'model juara':juara, 'MAPE h=5 (%)':round(skor[juara],3),
                     '2024':a[kol].iloc[-1], '2030':pk[5], '2035':pk[10],
                     'perubahan 2035 (%)':100*(pk[10]-a[kol].iloc[-1])/a[kol].iloc[-1]})
display(pd.DataFrame(ring_sek).round(3))

fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, (kol, lab) in zip(axes, SEKUNDER.items()):
    ax.plot(a.index, a[kol], color=PALETTE['primary'], lw=1.9, label='Aktual')
    ax.plot(np.r_[a.index[-1], tahun_depan], np.r_[a[kol].iloc[-1], proy_sek[kol]],
            color=PALETTE['danger'], lw=2, ls='--', label='Proyeksi')
    ax.axvline(a.index[-1], color=PALETTE['muted'], ls=':', lw=.9)
    ax.set_title(lab, fontsize=9.5); ax.set_xlabel('Tahun'); ax.legend(fontsize=7)
fig.suptitle('Proyeksi Seri Sekunder Panel Pembuka, sampai 2035', fontweight='bold', y=1.03)
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/17_seri_sekunder.svg', format='svg', bbox_inches='tight'); plt.show()

## 15. Mengapa CO₂ Alih Guna Lahan Tahunan Tidak Diramal

Seri ini adalah kandidat paling menggoda untuk pembuka narasi, tetapi harus ditolak. Bagian ini
mendokumentasikan alasannya secara kuantitatif — temuan negatif yang justru menyelamatkan naskah
dari klaim yang mudah dibantah.

In [ ]:
tolak = []
for kol, lab in [('luc','CO2 alih guna lahan tahunan'), ('pangan_ghg','GHG sistem pangan absolut'),
                 ('ch4','Metana (pembanding, dipakai)')]:
    y_t = np.log(a[kol].values.astype(float))
    for h in [3, 5, 10]:
        e = []
        for o in range(ORIGIN_MIN, len(y_t)-h+1):
            try:
                p = np.exp(MODEL_FN[BEST](y_t[:o], h))
                if np.all(np.isfinite(p)) and np.all(p > 0):
                    e.append(mape(a[kol].values[o:o+h], p))
            except Exception: pass
        tolak.append({'seri':lab, 'h':h, 'MAPE (%)':np.mean(e) if e else np.nan})
tb = pd.DataFrame(tolak).pivot(index='seri', columns='h', values='MAPE (%)').round(2)
tb.columns = [f'h={c}' for c in tb.columns]; display(tb)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(a.index, a['luc'], color=PALETTE['danger'], lw=1.8)
axes[0].set_title('CO2 alih guna lahan tahunan — bergejolak, MAPE > 20%', fontsize=10)
axes[1].plot(a.index, a['ch4'], color=PALETTE['primary'], lw=1.8)
axes[1].set_title('Metana — mulus, MAPE < 5%', fontsize=10)
for ax in axes: ax.set_xlabel('Tahun')
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/18_seri_ditolak.svg', format='svg', bbox_inches='tight'); plt.show()

kv_luc = a['luc'].pct_change().std(); kv_ch4 = a['ch4'].pct_change().std()
print(f'Simpangan baku perubahan tahunan: CO2 lahan {kv_luc:.4f} vs metana {kv_ch4:.4f} '
      f'({kv_luc/kv_ch4:.1f}x lebih bergejolak)')

## 16. Dekomposisi Skala vs Intensitas

Angka absolut yang naik dapat bersumber dari dua hal yang sangat berbeda: jumlah penduduk yang
bertambah, atau intensitas per orang yang memburuk. Pemisahan ini wajib dilakukan sebelum
menuliskan narasi urgensi, agar klaim yang dipakai tidak bertentangan dengan datanya sendiri.

In [ ]:
dek = []
for kol, lab in [('populasi_jt','Populasi'), ('ch4','Metana absolut'), ('ch4_pc','Metana per kapita'),
                 ('n2o','N2O absolut'), ('n2o_pc','N2O per kapita'),
                 ('luc','CO2 lahan absolut'), ('pangan_ghg','GHG pangan absolut')]:
    dek.append({'indikator':lab,
                '1961-2024 (%)':100*(a[kol].iloc[-1]-a[kol].iloc[0])/abs(a[kol].iloc[0]),
                '2000-2024 (%)':100*(a[kol].iloc[-1]-a[kol].loc[2000])/abs(a[kol].loc[2000])})
dd = pd.DataFrame(dek).round(1); display(dd)

fig, ax = plt.subplots(figsize=(9, 4.4))
w = [PALETTE['danger'] if v > 0 else PALETTE['secondary'] for v in dd['2000-2024 (%)']]
ax.barh(dd['indikator'], dd['2000-2024 (%)'], color=w)
ax.axvline(0, color='grey', lw=.9)
ax.set(xlabel='Perubahan 2000-2024 (%)',
       title='Yang naik dan yang turun: skala bertambah, intensitas per kapita membaik')
plt.tight_layout(); fig.savefig(f'{SVG_DIR}/19_skala_vs_intensitas.svg', format='svg', bbox_inches='tight'); plt.show()
print('Tafsir untuk naskah: emisi absolut naik karena SKALA (populasi +32%), bukan karena')
print('setiap orang makin boros -- intensitas per kapita justru membaik. Klaim urgensi harus')
print('dibangun di atas fakta ini, bukan melawannya.')

## 17. Ringkasan Angka untuk Infografis

In [ ]:
print('='*72); print('ANGKA SIAP PAKAI — PANEL PEMBUKA'); print('='*72)
print(f'\nModel juara metana        : {BEST}')
print(f'MAPE walk-forward h=5     : '
      f'{ringkas.loc["expanding"].xs(5, level="h").loc[BEST,"mape"]:.2f}%')
print(f'MASE h=5                  : '
      f'{ringkas.loc["expanding"].xs(5, level="h").loc[BEST,"mase"]:.3f}')
print(f'Anggota ensemble          : {", ".join(ANGGOTA)}')
print(f'\n--- Model dasar (Bagian 6-13, 12 kandidat) ---')
print(f'Proyeksi 2030             : {proyeksi.loc[2030,"ensemble"]:,.1f} Mt  '
      f'({100*(proyeksi.loc[2030,"ensemble"]-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}%)')
print(f'Proyeksi 2035             : {proyeksi.loc[2035,"ensemble"]:,.1f} Mt  '
      f'({100*(proyeksi.loc[2035,"ensemble"]-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}%)')
print(f'\n--- MODEL FINAL TERTUNING (Bagian 13B-13C, 266 konfigurasi) ---')
print(f'Aturan                    : h<=3 Drift polos ; h>=4 ensemble top-5')
print(f'Perbaikan pada fold OUTER : '
      f'{100*(out_df["OUTER terpilih"].mean()-out_df["OUTER Drift polos"].mean())/out_df["OUTER Drift polos"].mean():+.1f}%')
print(f'MAPE CV h=5 / h=10        : {cv_h[5]:.3f}% / {cv_h[10]:.3f}%')
print(f'\nMetana ASEAN 2024         : {a[Y_COL].iloc[-1]:,.1f} Mt CO2e')
print(f'Proyeksi 2030 (final)     : {proy_t.loc[2030,"ramalan"]:,.1f} Mt  '
      f'({100*(proy_t.loc[2030,"ramalan"]-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}%)')
print(f'Proyeksi 2035 (final)     : {proy_t.loc[2035,"ramalan"]:,.1f} Mt  '
      f'({100*(proy_t.loc[2035,"ramalan"]-a[Y_COL].iloc[-1])/a[Y_COL].iloc[-1]:+.1f}%)')
print(f'Selang 80% pada 2035      : {proy_t.loc[2035,"ci80_bawah"]:,.1f} - '
      f'{proy_t.loc[2035,"ci80_atas"]:,.1f} Mt')
print(f'\nPatahan struktural        : {BRK} (F={top.iloc[0].F:.2f}, p={top.iloc[0].p:.5f})')
print(f'Skema dummy terbaik       : {best_sk}')
print(f'Rekonsiliasi hirarkis     : bobot langsung {W_OPT[0]:.2f} / bottom-up {W_OPT[1]:.2f}')
print('\nSeri sekunder:'); display(pd.DataFrame(ring_sek).round(2))

## 18. Pembanding dengan statsmodels (opsional)

Sel ini hanya berjalan bila `statsmodels` terpasang, dan berfungsi memverifikasi bahwa
implementasi numpy di atas menghasilkan angka yang sepadan.

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller, kpss as sm_kpss
    from statsmodels.tsa.arima.model import ARIMA as SM_ARIMA
    from statsmodels.stats.diagnostic import acorr_ljungbox
    print(f'ADF  numpy = {adf(ly)[0]:.4f} | statsmodels = {adfuller(ly, autolag="AIC")[0]:.4f}')
    print(f'KPSS numpy = {kpss(ly):.4f} | statsmodels = {sm_kpss(ly, regression="c", nlags="auto")[0]:.4f}')
    sm = SM_ARIMA(ly, order=(orde[0], D_ORDER, orde[1])).fit()
    p_sm = np.exp(sm.forecast(11))
    print(f'\nProyeksi 2035 — numpy Hannan-Rissanen vs statsmodels MLE:')
    print(f'   numpy       = {np.exp(f_arima(ly, 11))[-1]:,.1f} Mt')
    print(f'   statsmodels = {p_sm[-1]:,.1f} Mt')
    print(f'\nLjung-Box statsmodels:'); display(acorr_ljungbox(resid, lags=[5,10], return_df=True).round(4))
except ImportError:
    print('statsmodels tidak terpasang — lewati pembanding.')